# Method 2 — Few-Shot Toxicity Classifier (41-Combo Coverage)

GPT-4.1 with static few-shot examples covering 41 label combinations.
Confidence scores (0-100).

In [11]:
# ============================================================
# IMPORTS
# ============================================================
import os, re, json, time, math, asyncio, hashlib
import numpy as np
import pandas as pd
import tiktoken

from openai import OpenAI, AsyncOpenAI
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score,
    hamming_loss, jaccard_score, multilabel_confusion_matrix,
    roc_auc_score
)


In [ ]:
# ============================================================
# CONFIG
# ============================================================

# ── Files ─────────────────────────────────────────────────────
BASE_DIR    = "/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment"
TEST_CSV    = f"{BASE_DIR}/stress_test_eval_set.csv"
OUTPUT_CSV  = f"{BASE_DIR}/predictions_few_shot41.csv"

# ── Models ────────────────────────────────────────────────────
GPT_MODEL   = "gpt-4.1"
EMBED_MODEL = "text-embedding-3-small"

# ── Labels ────────────────────────────────────────────────────
LABEL_COLS         = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
HARD_LABELS        = ["severe_toxic", "threat", "identity_hate"]
PRECISION_PRIORITY = ["severe_toxic", "threat", "insult"]

# ── Thresholds (applied to 0-100 confidence scores) ──────────
LABEL_THRESHOLDS = {
    "toxic": 25, "severe_toxic": 75, "obscene": 40,
    "threat": 40, "insult": 50, "identity_hate": 55,
}

# ── Async settings ────────────────────────────────────────────
ASYNC_CONCURRENCY = 8
CHECKPOINT_EVERY  = 500

# ── Pricing ───────────────────────────────────────────────────
GPT_INPUT_COST_PER_1K  = 0.00080
GPT_OUTPUT_COST_PER_1K = 0.00320
EMBED_COST_PER_1K      = 0.00002

# ── Reproducibility ───────────────────────────────────────────
GPT_SEED = 42

# ── Method info ───────────────────────────────────────────────
METHOD_NAME = "Few-Shot 41-Combo (GPT-4.1)"

# ── Clients ───────────────────────────────────────────────────
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key:
    raise ValueError("OPENAI_API_KEY not set. Run: export OPENAI_API_KEY='sk-...'")

client       = OpenAI(api_key=api_key)
async_client = AsyncOpenAI(api_key=api_key)

try:
    _enc = tiktoken.encoding_for_model(GPT_MODEL)
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")


In [13]:
# ============================================================
# UTILITIES
# ============================================================

def seconds_to_hms(s):
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    s = s % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

def count_tokens(text):
    if not isinstance(text, str) or not text:
        return 0
    return len(_enc.encode(text))

def clean_text(text, max_chars=12000):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"[\x01-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    return text[:max_chars] if len(text) > max_chars else text

def safe_for_api(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"[\x01-\x08\x0B\x0C\x0E-\x1F\x7F-\x9F]", " ", text)
    text = re.sub(r"\\{3,}", " ", text)
    text = re.sub(r"[\ud800-\udfff]", "", text)
    text = re.sub(r"\s{3,}", "  ", text).strip()
    return text

def safe_json_extract(raw):
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def extract_scores(raw):
    parsed = safe_json_extract(raw)
    if parsed is None:
        return None
    scores = parsed.get("scores", parsed)
    if not isinstance(scores, dict):
        return None
    result = {}
    for c in LABEL_COLS:
        try:
            result[c] = int(float(scores.get(c, 0)))
        except (ValueError, TypeError):
            result[c] = 0
    # If model returned binary 0/1 instead of 0-100, scale up
    if all(v in (0, 1) for v in result.values()):
        result = {c: v * 100 for c, v in result.items()}
    return result

def apply_thresholds(scores, thresholds=None):
    t = thresholds or LABEL_THRESHOLDS
    return {c: int(scores[c] >= t[c]) for c in LABEL_COLS}


In [14]:
# ============================================================
# RETRY WRAPPER
# ============================================================

def call_with_backoff(fn, *args, max_retries=5, base_delay=1.0, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower() or "timeout" in err.lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [backoff] attempt {attempt+1} — {delay:.1f}s")
                time.sleep(delay)
            elif "400" in err and attempt == 0:
                print(f"  [400-fix] sanitizing and retrying")
                if "messages" in kwargs:
                    kwargs["messages"] = [
                        {k: safe_for_api(v) if isinstance(v, str) else v for k, v in msg.items()}
                        for msg in kwargs["messages"]
                    ]
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


## Few-Shot Examples (41-Combo Coverage)

In [15]:
# ============================================================
# 41-COMBO REAL-DATA FEW-SHOT EXAMPLES (CURATED)
# Source: few_shot_pool.csv (151,571 comments)
# Each example represents one of the 41 unique label combinations
# in the Jigsaw dataset. Examples use REAL uncensored text from
# the pool to match the distribution the model will classify.
#
# Selection criteria:
#   - Real data (not synthetic)
#   - 40-250 chars (clear, not too long)
#   - High quality (not spammy/repetitive)
#   - Clear representative of its label combination
#   - Contrastive: similar combos have visibly different severity
# ============================================================

FEW_SHOT_EXAMPLES = [
    # ── 1. clean (pool: 136,195) ──
    ("Your rant above seems to be mainly about the fact that you don't like Gilbert, and seems to have little to do with improvement of this article.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 0, "identity_hate": 0}),

    # ── 2. toxic (pool: 5,374) ──
    ("Well you bloody should be. I have not been repeatedly blocked - only once and that was lies.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 0, "identity_hate": 0}),

    # ── 3. toxic+obscene+insult (pool: 3,617) ──
    ("Your'e a fucking nerd. And I feel real bad for your mother.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 1, "identity_hate": 0}),

    # ── 4. toxic+obscene (pool: 1,661) ──
    ("You know what ID like you to do? LEAVE ME THE FUCK ALONE!!!",
     {"toxic": 1, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 0, "identity_hate": 0}),

    # ── 5. toxic+insult (pool: 1,139) ──
    ("It's extremely comical listening to someone who runs around with a convicted felon, violent psycho terrorist punk and child abuser like Mordecai Levy pass judgement on anyone elses associations.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 1, "identity_hate": 0}),

    # ── 6. toxic+severe_toxic+obscene+insult (pool: 938) ──
    ("To faithlessfaggotboy: Suck my dick you goddamn gay-lovin piece of shit liberal. If I ever met you on the street I'd knock your empty head off your shoulders.",
     {"toxic": 1, "severe_toxic": 1, "obscene": 1, "threat": 0, "insult": 1, "identity_hate": 0}),

    # ── 7. toxic+obscene+insult+identity_hate (pool: 581) ──
    ("umm what was the delete for? u fuckin nigger monkey",
     {"toxic": 1, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 1, "identity_hate": 1}),

    # ── 8. obscene (pool: 299) ──
    ("Thank you, should I one day wish to yet again argue with a psychotic bitch I'll let you know. Meanwhile, please do stay off my talkpage.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 0, "identity_hate": 0}),

    # ── 9. insult (pool: 286) ──
    ("List of the least informed, abusive, imbeciles editing this site: Dionyseus, Ryan Delaney, Aitias, Fred Bauder",
     {"toxic": 0, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 1, "identity_hate": 0}),

    # ── 10. toxic+severe_toxic+obscene+insult+identity_hate (pool: 256) ──
    ("Hey Why the fuck would I vandalize my own article, dipshit? Keep your fucking jew-nose out of peoples goddamn business, got that asshole?",
     {"toxic": 1, "severe_toxic": 1, "obscene": 1, "threat": 0, "insult": 1, "identity_hate": 1}),

    # ── 11. obscene+insult (pool: 170) ──
    ("Wow, you're so clever. So smooth. Stop being an ass so we can compromise!",
     {"toxic": 0, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 1, "identity_hate": 0}),

    # ── 12. toxic+severe_toxic+obscene (pool: 148) ──
    ("you bunch of cock cucking, toilet scrubbing, tit licking, fanny kissing, arse wanking BASTARDS I HOPE YOU ALL ROT IN HELL",
     {"toxic": 1, "severe_toxic": 1, "obscene": 1, "threat": 0, "insult": 0, "identity_hate": 0}),

    # ── 13. toxic+identity_hate (pool: 132) ──
    ("Both of these two people are Gay kids and need to be Taught a Big Lesson from their Daddy so do not interrupt.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 0, "identity_hate": 1}),

    # ── 14. toxic+insult+identity_hate (pool: 128) ──
    ("Singing is one thing, but you also need to show that you're a smart person and not some illiterate redneck from the south.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 1, "identity_hate": 1}),

    # ── 15. toxic+obscene+threat+insult (pool: 127) ──
    ("Unblock me now you piece of shit! Or I will find you and kill you, slowly and painfully!",
     {"toxic": 1, "severe_toxic": 0, "obscene": 1, "threat": 1, "insult": 1, "identity_hate": 0}),

    # ── 16. toxic+threat (pool: 102) ──
    ("Don't ever show your face where I live and don't ever let our paths cross in this life or the next.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 1, "insult": 0, "identity_hate": 0}),

    # ── 17. toxic+severe_toxic+obscene+threat+insult (pool: 61) ──
    ("I Hope You Get Hit By A Truck. You filthy piece of worthless crap, just go kill your self human slime.",
     {"toxic": 1, "severe_toxic": 1, "obscene": 1, "threat": 1, "insult": 1, "identity_hate": 0}),

    # ── 18. identity_hate (pool: 52) ──
    ("jews the only people that do conspiring is the jews, they are constantly conducting conspiracies and hide behind their shield of anti-semitism and racist",
     {"toxic": 0, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 0, "identity_hate": 1}),

    # ── 19. toxic+obscene+threat+insult+identity_hate (pool: 51) ──
    ("I AM GONNA CUT OFF ZHANZHAO'S FUCKING INDIAN-CHINESE BASTARD BALLS. Zhanzhao is GAY!!! WAHAHAH",
     {"toxic": 1, "severe_toxic": 0, "obscene": 1, "threat": 1, "insult": 1, "identity_hate": 1}),

    # ── 20. toxic+severe_toxic (pool: 39) ──
    ("YOU LITTLE SHIT WHAT IS IT WITH YOU PAKIS ALL STICKING TOGETHER??????",
     {"toxic": 1, "severe_toxic": 1, "obscene": 0, "threat": 0, "insult": 0, "identity_hate": 0}),

    # ── 21. toxic+obscene+identity_hate (pool: 33) ──
    ("whatever you say gay boy. atleast i have a life and im not fucking unchanging shit on wikipedia",
     {"toxic": 1, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 0, "identity_hate": 1}),

    # ── 22. toxic+severe_toxic+obscene+threat+insult+identity_hate (pool: 28) ──
    ("fuck you honkey, why you hatin' on blacks? You fucking pussy geek nazi, I'll come to your house and rape your mom because I've got a big black dick in her tight white ass",
     {"toxic": 1, "severe_toxic": 1, "obscene": 1, "threat": 1, "insult": 1, "identity_hate": 1}),

    # ── 23. insult+identity_hate (pool: 27) ──
    ("That comment always comes from stupid americans who cannot visualise a world outside their borders.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 1, "identity_hate": 1}),

    # ── 24. threat (pool: 21) ──
    ("I shalt sever thy head at the neck. It will be dreadfully slow and painful, in accordance with Wikipedia policies.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 0, "threat": 1, "insult": 0, "identity_hate": 0}),

    # ── 25. obscene+insult+identity_hate (pool: 17) ──
    ("People who are athiests will go to he double hockey stick. Have fun there with the demons. You whore.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 1, "identity_hate": 1}),

    # ── 26. toxic+threat+insult (pool: 15) ──
    ("Hope he dies. Now this Atheist filth's wife has gone, he should follow soon and give us all a rest from him.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 1, "insult": 1, "identity_hate": 0}),

    # ── 27. toxic+severe_toxic+insult (pool: 13) ──
    ("Hey, you freaking retard. Don't call people like UnrealCoopNet Grawp when they are obviously not Grawp.",
     {"toxic": 1, "severe_toxic": 1, "obscene": 0, "threat": 0, "insult": 1, "identity_hate": 0}),

    # ── 28. toxic+obscene+threat (pool: 11) ──
    ("IF YOU REVERT MY FUCKIN EDITS METROS I WILL KILL YOU!",
     {"toxic": 1, "severe_toxic": 0, "obscene": 1, "threat": 1, "insult": 0, "identity_hate": 0}),

    # ── 29. toxic+severe_toxic+threat (pool: 11) ──
    ("I think New Yorkers should be allowed to have guns so they can shoot people like you.",
     {"toxic": 1, "severe_toxic": 1, "obscene": 0, "threat": 1, "insult": 0, "identity_hate": 0}),

    # ── 30. toxic+severe_toxic+insult+identity_hate (pool: 7) ──
    ("Remeber rahmatul lil alameen? You Sunni monkeys should remember this! Stop persecuting Ahmadis you terrorists!",
     {"toxic": 1, "severe_toxic": 1, "obscene": 0, "threat": 0, "insult": 1, "identity_hate": 1}),

    # ── 31. toxic+threat+identity_hate (pool: 7) ──
    ("Mmmm... yes... a pint of palestinian blood would be a fine dessert right now.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 1, "insult": 0, "identity_hate": 1}),

    # ── 32. toxic+severe_toxic+obscene+identity_hate (pool: 5) ──
    ("You Cunt Go fuck yourself you piece of shit. Don't make me tea beg your mama you faggot!!!",
     {"toxic": 1, "severe_toxic": 1, "obscene": 1, "threat": 0, "insult": 0, "identity_hate": 1}),

    # ── 33. toxic+severe_toxic+obscene+threat (pool: 4) ──
    ("DONT SEND ME A WARNING AGAIN OR I WILL RAPE YOUR FACE AND YOUR DAD BUM HOLE",
     {"toxic": 1, "severe_toxic": 1, "obscene": 1, "threat": 1, "insult": 0, "identity_hate": 0}),

    # ── 34. threat+insult (pool: 3) ──
    ("hey!!!!!!! thank you for re-editing my contributions it was really kind of you!!!!!!! i hope you get bowel cancer",
     {"toxic": 0, "severe_toxic": 0, "obscene": 0, "threat": 1, "insult": 1, "identity_hate": 0}),

    # ── 35. toxic+threat+insult+identity_hate (pool: 3) ──
    ("I think we should kill all the jews and homosexuals and handicaped people.",
     {"toxic": 1, "severe_toxic": 0, "obscene": 0, "threat": 1, "insult": 1, "identity_hate": 1}),

    # ── 36. obscene+identity_hate (pool: 3) ──
    ("Jews for Jesus Some Jews love Jesus in a way that can't be fully described with words... you know, in their anus.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 1, "threat": 0, "insult": 0, "identity_hate": 1}),

    # ── 37. toxic+severe_toxic+identity_hate (pool: 2) ──
    ("NIGGERS! Does really matter to label them? as long as they are dead.",
     {"toxic": 1, "severe_toxic": 1, "obscene": 0, "threat": 0, "insult": 0, "identity_hate": 1}),

    # ── 38. obscene+threat+insult (pool: 2) ──
    ("Treivas, Miller: you think you're safe over at Fire Island? We see you bitches are up to the same old same old.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 1, "threat": 1, "insult": 1, "identity_hate": 0}),

    # ── 39. toxic+severe_toxic+threat+insult (pool: 1) ──
    ("IM GOING TO KILL YOU ALL!!!!!!!!!!!!!!!!! I'm serious, you are all mean nasty idiots who deserve to die and I'm going to throw every single one of you in a shredder!!!!!!!",
     {"toxic": 1, "severe_toxic": 1, "obscene": 0, "threat": 1, "insult": 1, "identity_hate": 0}),

    # ── 40. obscene+threat (pool: 1) ──
    ("Kill Feldspar. Feldspar is a blathering idiot that should be publicly executed for his attempted use of force in place of honest debate.",
     {"toxic": 0, "severe_toxic": 0, "obscene": 1, "threat": 1, "insult": 0, "identity_hate": 0}),

    # ── 41. toxic+severe_toxic+threat+identity_hate (pool: 1) ──
    ("DEATH TO ARABS! FREE THE JEWISH HOMELAND! END ARAB COLONIALISM AND ARAB IMPERIALISM!",
     {"toxic": 1, "severe_toxic": 1, "obscene": 0, "threat": 1, "insult": 0, "identity_hate": 1}),
]

def format_few_shot_block():
    blocks = []
    for i, (comment, labels) in enumerate(FEW_SHOT_EXAMPLES, 1):
        # Convert binary labels to scores (0→0, 1→100) for consistency with score-based output
        scores = {k: v * 100 for k, v in labels.items()}
        blocks.append(f"Example {i}\nComment: {comment}\nScores: {json.dumps(scores)}")
    return "\n\n".join(blocks)

FEW_SHOT_BLOCK = format_few_shot_block()
print(f"Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")


Few-shot examples: 41


## Few-Shot Prompt Builder

In [16]:
def build_few_shot_prompt(row):
    """
    Build few-shot prompt from 41-combo examples.
    Returns (prompt, system_msg, embed_tokens).
    """
    comment = clean_text(str(row.comment_text))

    system_msg = "You are a precise toxicity classification assistant."

    json_template = (
        '{"scores": {"toxic": 0, "severe_toxic": 0, "obscene": 0, '
        '"threat": 0, "insult": 0, "identity_hate": 0}}'
    )

    prompt = f"""You are an expert multi-label toxicity classifier.

Labels and definitions:
- toxic: rude, hateful, or aggressive language
- severe_toxic: extreme abuse or dehumanising hostility (STRICTER than toxic; dehumanisation, calls for violence)
- obscene: explicit profanity or vulgar sexual language
- threat: real, specific implication of harm toward a person (NOT general anger or hyperbole)
- insult: direct personal attack on the person addressed
- identity_hate: attack targeting a protected GROUP identity (race, religion, gender, sexuality)

Decision rules:
1. severe_toxic is a strict subset of toxic.
2. threat requires clear intent — not just angry language.
3. identity_hate must target GROUP identity, not just insult an individual.
4. Multi-label output is allowed.

Reference examples (use these to calibrate your confidence scores):
{FEW_SHOT_BLOCK}

Classify this comment:
{comment}

Output ONLY this JSON with confidence scores 0-100 for each label and nothing else:
{json_template}"""

    return prompt, system_msg, 0

In [17]:
# ============================================================
# INFERENCE ENGINE (shared across methods 1-3)
# ============================================================

def gpt_classify(prompt, system_msg="You are a precise toxicity classification assistant."):
    """Call GPT and return raw response + token counts."""
    safe_prompt = safe_for_api(prompt)
    safe_sys    = safe_for_api(system_msg)
    resp = call_with_backoff(
        client.chat.completions.create,
        model=GPT_MODEL,
        temperature=0,
        seed=GPT_SEED,
        max_tokens=150,
        messages=[
            {"role": "system", "content": safe_sys},
            {"role": "user",   "content": safe_prompt},
        ]
    )
    raw     = resp.choices[0].message.content.strip()
    in_tok  = count_tokens(safe_prompt) + count_tokens(safe_sys)
    out_tok = count_tokens(raw)
    return raw, in_tok, out_tok

async def process_row(row, build_prompt_fn, semaphore):
    """Process a single row: build prompt, call GPT, extract scores, apply thresholds."""
    loop = asyncio.get_event_loop()
    row_start = time.perf_counter()

    prompt, system_msg, embed_tokens = build_prompt_fn(row)

    async with semaphore:
        raw, in_tok, out_tok = await loop.run_in_executor(
            None, gpt_classify, prompt, system_msg
        )

    scores = extract_scores(raw)
    if scores is None:
        # Fallback: simpler prompt
        fallback = (
            f"Classify this comment for toxicity. Return ONLY valid JSON with integer scores 0-100:\n"
            f"{clean_text(str(row.comment_text))}\n\n"
            f'{{"scores":{{"toxic":0,"severe_toxic":0,"obscene":0,"threat":0,"insult":0,"identity_hate":0}}}}'
        )
        raw2, in2, out2 = await loop.run_in_executor(
            None, gpt_classify, fallback, "You are a JSON-only classifier."
        )
        in_tok += in2; out_tok += out2
        scores = extract_scores(raw2) or {c: 0 for c in LABEL_COLS}
        raw = raw2

    pred = apply_thresholds(scores)
    row_elapsed = time.perf_counter() - row_start

    out = {
        "id":                str(getattr(row, "id", row.Index)),
        "comment_text":      row.comment_text,
        "raw_response":      raw,
        "scores":            json.dumps(scores, ensure_ascii=False),
        "embed_tokens":      embed_tokens,
        "gpt_input_tokens":  in_tok,
        "gpt_output_tokens": out_tok,
        "row_runtime_sec":   row_elapsed,
    }
    for c in LABEL_COLS:
        out[f"true_{c}"] = int(getattr(row, c, 0))
        out[f"pred_{c}"] = int(pred[c])

    return out

async def run_inference(df, build_prompt_fn):
    """Run async inference over all rows with checkpoint/resume."""
    t0 = time.perf_counter()

    print("=" * 72)
    print(f"INFERENCE — {METHOD_NAME}")
    print(f"  Model     : {GPT_MODEL}")
    print(f"  Rows      : {len(df):,}")
    print(f"  Concurrency: {ASYNC_CONCURRENCY}")
    print("=" * 72)

    # Resume support
    if os.path.exists(OUTPUT_CSV):
        done_df  = pd.read_csv(OUTPUT_CSV)
        done_ids = set(done_df["id"].astype(str))
        rows     = done_df.to_dict(orient="records")
        print(f"Resuming from {len(rows):,} completed rows")
    else:
        done_ids = set()
        rows     = []

    pending = [r for r in df.itertuples() if str(getattr(r, "id", r.Index)) not in done_ids]
    print(f"Rows to process: {len(pending):,}")

    semaphore = asyncio.Semaphore(ASYNC_CONCURRENCY)
    tasks = [process_row(row, build_prompt_fn, semaphore) for row in pending]

    completed = 0
    for coro in asyncio.as_completed(tasks):
        result = await coro
        rows.append(result)
        completed += 1
        if completed % CHECKPOINT_EVERY == 0:
            pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
            cost = (
                sum(r.get("gpt_input_tokens", 0) for r in rows) / 1000 * GPT_INPUT_COST_PER_1K +
                sum(r.get("gpt_output_tokens", 0) for r in rows) / 1000 * GPT_OUTPUT_COST_PER_1K +
                sum(r.get("embed_tokens", 0) for r in rows) / 1000 * EMBED_COST_PER_1K
            )
            print(f"  [{len(rows):>6,}/{len(df):<6,}] cost~${cost:.4f}")

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

    elapsed = time.perf_counter() - t0
    print(f"\nInference complete. {len(rows):,} rows | {seconds_to_hms(elapsed)}")
    print(f"Saved to: {OUTPUT_CSV}")
    print("=" * 72)


In [18]:
# ============================================================
# UNIFIED EVALUATION (matches v4 output format)
# ============================================================

def evaluate_predictions(csv_path=None, method_name=None):
    """
    Unified evaluation matching v4 format.
    Works for all methods — zero-shot, few-shot, RAG static, RAG label-aware.
    """
    csv_file = csv_path or OUTPUT_CSV
    _method  = method_name or METHOD_NAME
    eval_start = time.perf_counter()

    if not os.path.exists(csv_file):
        raise FileNotFoundError(f"{csv_file} not found.")

    df = pd.read_csv(csv_file)
    y_true = df[[f"true_{c}" for c in LABEL_COLS]].values.astype(int)
    y_pred = df[[f"pred_{c}" for c in LABEL_COLS]].values.astype(int)

    # ── Global metrics ────────────────────────────────────────
    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    exact_acc  = accuracy_score(y_true, y_pred)
    h_loss     = hamming_loss(y_true, y_pred)
    jacc_macro = jaccard_score(y_true, y_pred, average="macro", zero_division=0)
    jacc_samp  = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

    # ── Per-label metrics ─────────────────────────────────────
    per_p, per_r, per_f1, per_sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    label_acc = {}
    for i, label in enumerate(LABEL_COLS):
        label_acc[label] = float(np.mean(y_true[:, i] == y_pred[:, i]))

    # ── ROC-AUC ───────────────────────────────────────────────
    has_scores = "scores" in df.columns
    per_auc = {}
    micro_auc = None
    macro_auc = None

    if has_scores:
        score_matrix = np.zeros((len(df), len(LABEL_COLS)), dtype=float)
        for j, label in enumerate(LABEL_COLS):
            score_matrix[:, j] = df["scores"].apply(
                lambda s, l=label: json.loads(s).get(l, 0) if isinstance(s, str) else 0
            ).values.astype(float)

        for j, label in enumerate(LABEL_COLS):
            if y_true[:, j].sum() > 0 and y_true[:, j].sum() < len(y_true):
                per_auc[label] = roc_auc_score(y_true[:, j], score_matrix[:, j])
            else:
                per_auc[label] = float("nan")

        valid_labels = [j for j, l in enumerate(LABEL_COLS) if not np.isnan(per_auc.get(l, float("nan")))]
        if valid_labels:
            try:
                micro_auc = roc_auc_score(y_true[:, valid_labels], score_matrix[:, valid_labels], average="micro")
            except ValueError:
                micro_auc = None
            macro_auc = float(np.nanmean([per_auc[LABEL_COLS[j]] for j in valid_labels]))

    # ── Operational metrics ───────────────────────────────────
    gpt_in_tok  = int(df["gpt_input_tokens"].fillna(0).sum())  if "gpt_input_tokens"  in df.columns else 0
    gpt_out_tok = int(df["gpt_output_tokens"].fillna(0).sum()) if "gpt_output_tokens" in df.columns else 0
    embed_tok   = int(df["embed_tokens"].fillna(0).sum())       if "embed_tokens"      in df.columns else 0
    mean_rt     = float(df["row_runtime_sec"].fillna(0).mean()) if "row_runtime_sec"   in df.columns else 0.0
    total_rt    = float(df["row_runtime_sec"].fillna(0).sum())  if "row_runtime_sec"   in df.columns else 0.0

    embed_cost = (embed_tok / 1000)   * EMBED_COST_PER_1K
    gin_cost   = (gpt_in_tok / 1000)  * GPT_INPUT_COST_PER_1K
    gout_cost  = (gpt_out_tok / 1000) * GPT_OUTPUT_COST_PER_1K
    total_cost = embed_cost + gin_cost + gout_cost

    cost_per_sample = total_cost / len(df) if len(df) > 0 else 0.0
    throughput      = len(df) / total_rt if total_rt > 0 else 0.0

    # ── Print report ──────────────────────────────────────────
    W = 78
    print(f"\n{'=' * W}")
    print(f"   {_method.upper()} — EVAL REPORT")
    print(f"{'=' * W}")
    print(f"  Generated   : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  GPT model   : {GPT_MODEL}")
    print(f"  Method      : {_method}")
    print(f"  Rows eval   : {len(df):,}")
    print()

    print("── GLOBAL METRICS ──")
    print(f"  Micro     Prec={micro_p:.4f}  Rec={micro_r:.4f}  F1={micro_f1:.4f}")
    print(f"  Macro     Prec={macro_p:.4f}  Rec={macro_r:.4f}  F1={macro_f1:.4f}")
    print(f"  Weighted  Prec={weighted_p:.4f}  Rec={weighted_r:.4f}  F1={weighted_f1:.4f}")
    print(f"  Exact Match Acc  : {exact_acc:.4f}")
    print(f"  Hamming Loss     : {h_loss:.4f}")
    print(f"  Jaccard(macro)   : {jacc_macro:.4f}")
    print(f"  Jaccard(samples) : {jacc_samp:.4f}")
    if micro_auc is not None: print(f"  ROC-AUC(micro)   : {micro_auc:.4f}")
    if macro_auc is not None: print(f"  ROC-AUC(macro)   : {macro_auc:.4f}")
    print()

    print("── PER-LABEL METRICS ──\n")
    hdr = f"  {'Label':<18}{'Prec':>8}{'Rec':>8}{'F1':>8}{'AUC':>8}{'Acc':>8}{'TP':>6}{'FP':>6}{'FN':>6}{'TN':>6}{'Supp':>6}"
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))
    for i, label in enumerate(LABEL_COLS):
        tn, fp, fn, tp = mcm[i].ravel()
        star = " ★" if label in PRECISION_PRIORITY else ""
        auc_str = f"{per_auc[label]:.4f}" if label in per_auc and not np.isnan(per_auc.get(label, float('nan'))) else "  n/a "
        print(f"  {label + star:<18}{per_p[i]:>8.4f}{per_r[i]:>8.4f}{per_f1[i]:>8.4f}{auc_str:>8}{label_acc[label]:>8.4f}{tp:>6}{fp:>6}{fn:>6}{tn:>6}{int(per_sup[i]):>6}")
    print(f"\n  Note: Label-wise Acc can be misleading on imbalanced labels — F1 and AUC are more reliable.")
    print()

    print("── OPERATIONAL METRICS ──")
    print(f"  Mean latency      : {mean_rt:.4f}s per row")
    print(f"  Throughput        : {throughput:.2f} rows/sec")
    print()

    print("── COST & TIME ──")
    if embed_tok > 0:
        print(f"  Embed tokens    : {embed_tok:,}   cost ~${embed_cost:.6f}")
    print(f"  GPT input tok.  : {gpt_in_tok:,}   cost ~${gin_cost:.6f}")
    print(f"  GPT output tok. : {gpt_out_tok:,}   cost ~${gout_cost:.6f}")
    print(f"  Total cost      : ${total_cost:.6f}")
    print(f"  Cost per sample : ${cost_per_sample:.6f}")
    print(f"  Eval compute    : {seconds_to_hms(time.perf_counter() - eval_start)}")
    print()
    print("✅ Evaluation complete.")
    print("=" * W)

    return {
        "method": _method, "n_rows": len(df),
        "micro_f1": micro_f1, "macro_f1": macro_f1, "weighted_f1": weighted_f1,
        "exact_match_acc": exact_acc, "hamming_loss": h_loss,
        "jaccard_samples": jacc_samp, "roc_auc_macro": macro_auc,
        "cost_per_sample": cost_per_sample, "throughput": throughput, "mean_latency": mean_rt,
    }


## Run Pipeline

In [19]:
# Load data
df = pd.read_csv(TEST_CSV).dropna(subset=["comment_text"]).copy()
df["comment_text"] = df["comment_text"].astype(str)
for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
if "id" not in df.columns:
    df["id"] = np.arange(len(df)).astype(str)
else:
    df["id"] = df["id"].astype(str)

print(f"Loaded {len(df):,} rows from {TEST_CSV}")


Loaded 8,000 rows from /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/stress_test_eval_set.csv


In [20]:
# ── Run inference + evaluation ─────────────────────────────────
await run_inference(df, build_few_shot_prompt)

# ── Evaluate (ROC-AUC included automatically) ─────────────────
print()
result = evaluate_predictions()

INFERENCE — Few-Shot 41-Combo (GPT-4.1)
  Model     : gpt-4.1
  Rows      : 8,000
  Concurrency: 8
Rows to process: 8,000
  [   500/8,000 ] cost~$1.5044
  [ 1,000/8,000 ] cost~$3.0115
  [ 1,500/8,000 ] cost~$4.5169
  [ 2,000/8,000 ] cost~$6.0203
  [ 2,500/8,000 ] cost~$7.5265
  [ 3,000/8,000 ] cost~$9.0272
  [ 3,500/8,000 ] cost~$10.5368
  [ 4,000/8,000 ] cost~$12.0424
  [ 4,500/8,000 ] cost~$13.5475
  [ 5,000/8,000 ] cost~$15.0513
  [ 5,500/8,000 ] cost~$16.5617
  [ 6,000/8,000 ] cost~$18.0634
  [ 6,500/8,000 ] cost~$19.5668
  [ 7,000/8,000 ] cost~$21.0707
  [ 7,500/8,000 ] cost~$22.5818
  [ 8,000/8,000 ] cost~$24.0887

Inference complete. 8,000 rows | 00:17:56.09
Saved to: /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/predictions_few_shot41.csv


   FEW-SHOT 41-COMBO (GPT-4.1) — EVAL REPORT
  Generated   : 2026-03-14 09:45:21
  GPT model   : gpt-4.1
  Method      : Few-Shot 41-Combo (GPT-4.1)
  Rows eval   : 8,000

── GLOBAL METRICS ──
  Micro  